# HimRaksha - raw Sentinel-1 GRD processing (Colab)
Downloads the original Sentinel-1 GRD scenes from Copernicus Data Space and processes them with our own chain:
precise orbit -> thermal noise removal -> border noise removal -> calibration (sigma0) -> subset -> terrain correction (Copernicus DEM, 10 m, UTM) -> mosaic -> dB.

Output: `s1_orbit<NN>_<direction>.tif` with bands pre_VV, pre_VH, post_VV, post_VH (dB) - the input of `01_mapping.py`.

Run top to bottom. Needs a free Copernicus Data Space account. Runtime: ~10 min SNAP install + ~10-20 min per scene.

In [ ]:
# 1) Get the code: upload pipeline/grd_preprocess.py from your clone of the repo
import os
os.makedirs('pipeline', exist_ok=True)
if not os.path.exists('pipeline/grd_preprocess.py'):
    from google.colab import files
    for name in files.upload():
        os.replace(name, 'pipeline/grd_preprocess.py')
import sys; sys.path.insert(0, 'pipeline')
!pip install -q rasterio pyproj
import grd_preprocess as g
print('code loaded')

In [ ]:
# 2) Install ESA SNAP (one time per Colab session, ~5-10 min)
GPT = g.install_snap()
!{GPT} -h | head -3

In [ ]:
# 3) Copernicus Data Space login - typed here, never saved in the notebook
import os, getpass
if not os.environ.get('CDSE_USER'):
    os.environ['CDSE_USER'] = input('Copernicus Data Space email: ').strip()
    os.environ['CDSE_PASSWORD'] = getpass.getpass('Password: ')
g.cdse_token(); print('login OK')

In [ ]:
# 4) Area and orbit pairs (Trishuli defaults = the pairs the mapping step chose)
BBOX = [85.10, 27.90, 85.45, 28.32]          # W, S, E, N
PAIRS = [
    {'orbit': 85, 'direction': 'ASCENDING',  'pre': '2026-08-16', 'post': '2026-08-28'},
    {'orbit': 19, 'direction': 'DESCENDING', 'pre': '2026-08-24', 'post': '2026-09-05'},
]
OUT = 'run_trishuli_grd'
for p in PAIRS:
    for when in ('pre', 'post'):
        sl = g.search_slices(BBOX, p['orbit'], p['direction'], p[when])
        print(p['orbit'], p['direction'], when, p[when], '->', len(sl), 'slice(s)', sum(s['size_mb'] for s in sl), 'MB')

In [ ]:
# 5) Download + process (cached: re-running skips finished steps)
outputs = []
for p in PAIRS:
    outputs.append(g.build_pair(BBOX, p, OUT, cache_dir='grd_cache', gpt=GPT))
outputs

In [ ]:
# 6) Look at the result
import rasterio, numpy as np, matplotlib.pyplot as plt
fig, ax = plt.subplots(len(outputs), 2, figsize=(12, 6 * len(outputs)), squeeze=False)
for k, f in enumerate(outputs):
    with rasterio.open(f) as s:
        pre, post = s.read(1), s.read(3)
    ax[k, 0].imshow(pre, cmap='gray', vmin=-25, vmax=0); ax[k, 0].set_title(f'{os.path.basename(f)}  pre VV')
    ax[k, 1].imshow(post, cmap='gray', vmin=-25, vmax=0); ax[k, 1].set_title('post VV')
    for a in ax[k]: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 7) OPTIONAL check vs the old openEO output (upload s1_orbit85_ascending.tif etc. from the earlier run zip)
from google.colab import files
os.makedirs('openeo', exist_ok=True)
for name in files.upload():
    os.replace(name, f'openeo/{name}')
results = {}
for f in outputs:
    old = f'openeo/{os.path.basename(f)}'
    if os.path.exists(old):
        tag = os.path.basename(f).replace('.tif', '')
        results[tag] = g.compare(f, old, f'{OUT}/grd_vs_openeo_{tag}.json', f'{OUT}/grd_vs_openeo_{tag}.png')
results

In [ ]:
# 8) Download the results (stacks + checks; the big raw zips stay in grd_cache)
import shutil
shutil.make_archive(OUT, 'zip', OUT)
from google.colab import files
files.download(OUT + '.zip')